# Chapter 3 · Prompting and Context Engineering
**Mastering Agentic AI** — Manning Publications



## 3.0 · Setup

See `appendix_a_api_keys.md` for DSPy/OpenAI key setup if using Section 3.5.

In [ ]:
# !pip install anthropic dspy-ai pydantic python-dotenv
import os, json, re
from anthropic import Anthropic
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "See appendix_a_api_keys.md"
client = Anthropic()
MODEL = "claude-opus-4-5"
print("Ready")

## 3.1 · Agent Profiling: The Five Structural Questions

| Question | Why it matters |
|---|---|
| What is your role? | Anchors reasoning persona |
| What is the goal? | Defines stopping condition |
| What tools can you use? | Prevents hallucination of capabilities |
| What should you avoid? | Hard guardrails |
| What format should you return? | Reliable downstream integration |

In [ ]:
SYSTEM_PROMPT = """[ROLE]
You are an AI Diet Coach with formal training in nutrition science.

[SKILL — Nutrition Assessment Protocol]
When conducting a full assessment, follow these four steps:
  Step 1 — Establish Baseline: eating pattern, restrictions, goals, activity.
  Step 2 — Identify Deficits: protein (<0.8g/kg), fibre (<25g/day), micronutrients.
             Flag deficits explicitly — do not soften findings.
  Step 3 — Prioritise Changes: high-impact, low-effort first.
  Step 4 — Set One Measurable Goal: exactly one concrete, time-bound goal.

[CONSTRAINTS]
Never diagnose medical conditions.
Refer to a registered dietitian for therapeutic diets.

[FORMAT]
End assessments with 'Your Goal This Week:' section."""
print(f"Layers: ROLE + SKILL + CONSTRAINTS + FORMAT = {len(SYSTEM_PROMPT)} chars")

## 3.2 · Tools vs. Skills

In [ ]:
print("""
TOOLS (Chapter 2)              SKILLS (Chapter 3)
─────────────────────────────────────────────────
Nature    : Declarative (what)  vs  Procedural (how)
Defined   : JSON Schema + fn    vs  Natural language
Executed  : Your runtime        vs  The model
Gives     : Data access         vs  Judgment
Example   : lookup → 165 cal    vs  Assess → deficits → one goal
Missing   : Floods with numbers vs  Advice with no grounding
Together  : Grounded, strategic nutrition coaching
""")

## 3.3 · Structured Outputs with Pydantic

In [ ]:
class NutritionAssessment(BaseModel):
    baseline_summary : str       = Field(..., description="Summary of current eating pattern")
    deficits         : List[str] = Field(..., description="Identified nutritional deficits")
    priority_actions : List[str] = Field(..., description="Ranked recommended changes")
    goal_this_week   : str       = Field(..., description="Single concrete, time-bound goal")

print("Schema:", list(NutritionAssessment.model_fields.keys()))

In [ ]:
def run_structured_assessment(user_profile: dict) -> NutritionAssessment:
    "Skill-guided assessment returning a validated Pydantic object."
    prompt = (
        f"User profile: {json.dumps(user_profile, indent=2)}\n\n"
        "Conduct a four-step nutrition assessment (Baseline, Deficits, Priorities, Goal).\n"
        "Return ONLY valid JSON matching this schema (no markdown, no extra keys):\n"
        + json.dumps(NutritionAssessment.model_json_schema(), indent=2)
    )
    response = client.messages.create(
        model=MODEL, max_tokens=1024, system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = re.sub(r'^```(?:json)?\n?', '', response.content[0].text.strip()).rstrip('`').strip()
    return NutritionAssessment.model_validate_json(raw)

profile = {"name": "Alex", "age": 30, "weight_kg": 78, "goal": "lose 5kg",
           "typical_diet": "skips breakfast, large lunch, late dinner, drinks 1L water/day"}
assessment = run_structured_assessment(profile)
print("Deficits:      ", assessment.deficits)
print("Goal this week:", assessment.goal_this_week)

## 3.4 · Context Window Composition

In [ ]:
def build_context_window(user_message, meal_history=None, user_profile=None):
    """
    Layer context deliberately. Order = relevance.
    Most important context goes closest to the final message.
    """
    messages = []
    if user_profile:
        messages += [
            {"role": "user",      "content": f"[Context] Profile: {json.dumps(user_profile)}"},
            {"role": "assistant", "content": "Understood — profile loaded."}
        ]
    if meal_history:
        recent = meal_history[-3:]  # token budget: last 3 only
        note = "Recent meals: " + ", ".join(m["food"] for m in recent)
        messages += [
            {"role": "user",      "content": note},
            {"role": "assistant", "content": "Factoring in recent meals."}
        ]
    messages.append({"role": "user", "content": user_message})
    return messages

profile = {"name": "Alex", "weight_kg": 78, "goal": "lose weight"}
history = [{"food": "oats"}, {"food": "chicken"}, {"food": "apple"}]
msgs = build_context_window("Give me a nutrition assessment.", history, profile)
print(f"Context window: {len(msgs)} turns")

## 3.5 · Chapter Summary

| Concept | What we built |
|---|---|
| Five structural questions | Minimal agent profile spec |
| SKILL.md | 4-step Nutrition Assessment Protocol |
| Pydantic schema | Validated structured output |
| Context builder | Deliberate layering: profile → history → message |

**Next:** Chapter 4 — Tools and Interoperability.

---
*Mastering Agentic AI · Manning Publications*